<a href="https://colab.research.google.com/github/sadikinisaac/AIML/blob/main/sgstraitsvesselmovement1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGAPORE STRAIT — VESSEL TRAFFIC ANIMATION
# Free basemap via OpenStreetMap + contextily | 3-min MP4 | 102 vessels
# ═══════════════════════════════════════════════════════════════════════════════

# ── Install ──────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'contextily', 'pyproj', 'numpy', 'matplotlib', 'tqdm'], check=True)
subprocess.run(['apt-get', 'install', '-q', '-y', 'ffmpeg'], check=True)

# ── Imports ───────────────────────────────────────────────────────────────────
import io, os, math, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.animation import FFMpegWriter
from pyproj import Transformer
import contextily as ctx
from tqdm.auto import tqdm
from google.colab import files

# ═══════════════════════════════════════════════════════════════════════════════
# 1. MAP EXTENT  (Web Mercator EPSG:3857)
# ═══════════════════════════════════════════════════════════════════════════════

LON_MIN, LON_MAX = 103.55, 104.35
LAT_MIN, LAT_MAX = 1.10,  1.45

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
xmin, ymin = transformer.transform(LON_MIN, LAT_MIN)
xmax, ymax = transformer.transform(LON_MAX, LAT_MAX)

def to_merc(lons, lats):
    return transformer.transform(lons, lats)

# ═══════════════════════════════════════════════════════════════════════════════
# 2. FETCH & CACHE BASEMAP  (OpenStreetMap satellite — Esri World Imagery)
# ═══════════════════════════════════════════════════════════════════════════════

MAP_CACHE = '/tmp/sg_basemap_osm.npz'

if os.path.exists(MAP_CACHE):
    data      = np.load(MAP_CACHE)
    basemap   = data['img']
    bm_extent = list(data['extent'])
    print(f'✅  Loaded cached basemap  {basemap.shape[1]}×{basemap.shape[0]} px')
else:
    print('⬇️  Downloading basemap (Esri World Imagery via contextily)…')
    fig_tmp, ax_tmp = plt.subplots(figsize=(12, 7))
    ax_tmp.set_xlim(xmin, xmax)
    ax_tmp.set_ylim(ymin, ymax)
    ctx.add_basemap(ax_tmp, source=ctx.providers.Esri.WorldImagery, zoom=11, crs='EPSG:3857')
    # Extract the image artist
    imgs = [c for c in ax_tmp.get_children()
            if isinstance(c, matplotlib.image.AxesImage)]
    bm_img_artist = imgs[0]
    basemap   = (bm_img_artist.get_array() * 255).astype(np.uint8) \
                if bm_img_artist.get_array().max() <= 1.0 \
                else bm_img_artist.get_array().astype(np.uint8)
    bm_extent = list(bm_img_artist.get_extent())   # [x0, x1, y0, y1]
    plt.close(fig_tmp)
    np.savez_compressed(MAP_CACHE, img=basemap, extent=np.array(bm_extent))
    print(f'✅  Basemap saved  {basemap.shape[1]}×{basemap.shape[0]} px')

# ═══════════════════════════════════════════════════════════════════════════════
# 3. ANIMATION PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════

DURATION_S = 180
FPS        = 24
N_FRAMES   = DURATION_S * FPS    # 4 320
TRAIL      = FPS * 6             # 6-second wake trail
SIM_HOURS  = 12.0
SIM_START  = 6.0                 # simulation starts at 06:00 SGT

print(f'Total frames: {N_FRAMES}  ({DURATION_S}s × {FPS}fps)  |  102 vessels')

# ═══════════════════════════════════════════════════════════════════════════════
# 4. VESSEL TRACK GENERATION
# ═══════════════════════════════════════════════════════════════════════════════

rng = np.random.default_rng(42)

def make_track(lon0, lat0, lon1, lat1,
               spread_lon=0.008, spread_lat=0.006,
               speed_var=0.15, stagger=0, wobble=0.0005):
    dlon = rng.uniform(-spread_lon, spread_lon)
    dlat = rng.uniform(-spread_lat, spread_lat)
    speed = rng.normal(1.0, speed_var, N_FRAMES).clip(0.3, 2.0)
    cum   = np.cumsum(speed)
    t     = (cum - cum[0]) / (cum[-1] - cum[0])
    lon   = (lon0 + dlon) + t * (lon1 - lon0)
    lat   = (lat0 + dlat) + t * (lat1 - lat0)
    # lateral sinusoidal wobble
    phase = rng.uniform(0, 2 * np.pi)
    freq  = rng.uniform(0.3, 0.8)
    side  = np.sin(2 * np.pi * freq * t + phase) * wobble
    dx, dy = lon1 - lon0, lat1 - lat0
    norm = math.hypot(dx, dy) or 1
    lon += side * (-dy / norm)
    lat += side * ( dx / norm)
    if stagger > 0:
        lon = np.roll(lon, stagger); lon[:stagger] = np.nan
        lat = np.roll(lat, stagger); lat[:stagger] = np.nan
    return lon, lat

vessels = []

# Eastbound TSS (30)
for _ in range(30):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.58, 1.168, 104.28, 1.178,
                      spread_lon=0.0, spread_lat=0.012, stagger=stag)
    vessels.append({'type': 'eastbound', 'lon': l, 'lat': a})

# Westbound TSS (30)
for _ in range(30):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(104.28, 1.210, 103.58, 1.200,
                      spread_lon=0.0, spread_lat=0.012, stagger=stag)
    vessels.append({'type': 'westbound', 'lon': l, 'lat': a})

# SW / Philip Channel crossing (10)
for _ in range(10):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.58, 1.230, 103.88, 1.190,
                      spread_lon=0.01, spread_lat=0.008, stagger=stag)
    vessels.append({'type': 'crossing', 'lon': l, 'lat': a})

# Northbound – Changi / Johor Strait (6)
for _ in range(6):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.95 + rng.uniform(-0.04, 0.04), 1.155,
                      103.93 + rng.uniform(-0.04, 0.04), 1.330,
                      spread_lon=0.015, spread_lat=0.0, stagger=stag)
    vessels.append({'type': 'northbound', 'lon': l, 'lat': a})

# Southbound – Jurong / Keppel (6)
for _ in range(6):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.73 + rng.uniform(-0.04, 0.04), 1.330,
                      103.71 + rng.uniform(-0.04, 0.04), 1.155,
                      spread_lon=0.015, spread_lat=0.0, stagger=stag)
    vessels.append({'type': 'southbound', 'lon': l, 'lat': a})

# Anchored – Eastern Anchorage (12)
for _ in range(12):
    alon = rng.uniform(104.00, 104.20)
    alat = rng.uniform(1.220, 1.270)
    dl = np.cumsum(rng.normal(0, 0.00005, N_FRAMES))
    da = np.cumsum(rng.normal(0, 0.00003, N_FRAMES))
    vessels.append({'type': 'anchored', 'lon': alon + dl, 'lat': alat + da})

# Anchored – Western Anchorage (8)
for _ in range(8):
    alon = rng.uniform(103.58, 103.80)
    alat = rng.uniform(1.155, 1.210)
    dl = np.cumsum(rng.normal(0, 0.00005, N_FRAMES))
    da = np.cumsum(rng.normal(0, 0.00003, N_FRAMES))
    vessels.append({'type': 'anchored', 'lon': alon + dl, 'lat': alat + da})

print(f'Vessels spawned: {len(vessels)}')

# ═══════════════════════════════════════════════════════════════════════════════
# 5. PRE-COMPUTE MERCATOR COORDS  (skip repeated transformer calls per frame)
# ═══════════════════════════════════════════════════════════════════════════════

print('⚙️  Pre-computing Mercator coordinates…')
merc_coords = []
for v in vessels:
    lons, lats = v['lon'], v['lat']
    valid = ~np.isnan(lons)
    xs = np.full(N_FRAMES, np.nan)
    ys = np.full(N_FRAMES, np.nan)
    if valid.any():
        xs[valid], ys[valid] = to_merc(lons[valid], lats[valid])
    merc_coords.append((xs, ys))
print('✅  Done')

# ═══════════════════════════════════════════════════════════════════════════════
# 6. STYLE MAP
# ═══════════════════════════════════════════════════════════════════════════════

STYLE = {
    'eastbound':  {'color': '#00cfff', 'size': 28, 'marker': '>', 'zorder': 5},
    'westbound':  {'color': '#ff6b35', 'size': 28, 'marker': '<', 'zorder': 5},
    'crossing':   {'color': '#ffe066', 'size': 22, 'marker': '^', 'zorder': 4},
    'northbound': {'color': '#a0ff80', 'size': 22, 'marker': '^', 'zorder': 4},
    'southbound': {'color': '#ff80d5', 'size': 22, 'marker': 'v', 'zorder': 4},
    'anchored':   {'color': '#ffdd00', 'size': 20, 'marker': 's', 'zorder': 3},
}

# ═══════════════════════════════════════════════════════════════════════════════
# 7. FIGURE SETUP
# ═══════════════════════════════════════════════════════════════════════════════

DPI = 110
fig, ax = plt.subplots(figsize=(1280 / DPI, 720 / DPI), dpi=DPI, facecolor='black')
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.axis('off')

# Basemap
ax.imshow(
    basemap,
    extent=bm_extent,          # [x0, x1, y0, y1] in Mercator
    origin='upper',
    aspect='auto',
    zorder=0
)

# Scatter artists
scatters = {}
for vtype, st in STYLE.items():
    scatters[vtype] = ax.scatter(
        [], [], s=st['size'], c=st['color'],
        marker=st['marker'], zorder=st['zorder'],
        edgecolors='white', linewidths=0.3, alpha=0.92
    )

trail_lines = []

# HUD — title
ax.text(0.5, 0.977,
        'SINGAPORE STRAIT — VESSEL TRAFFIC',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=13, fontweight='bold', color='white', family='monospace',
        path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=10)

# HUD — clock
clock_txt = ax.text(
    0.013, 0.977, '',
    transform=ax.transAxes, ha='left', va='top',
    fontsize=9, color='#a0ffcc', family='monospace',
    path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=10)

# HUD — counts
count_txt = ax.text(
    0.987, 0.977, '',
    transform=ax.transAxes, ha='right', va='top',
    fontsize=8.5, color='white', family='monospace',
    path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=10)

# Legend
legend_items = [
    mpatches.Patch(color='#00cfff', label='Eastbound'),
    mpatches.Patch(color='#ff6b35', label='Westbound'),
    mpatches.Patch(color='#ffe066', label='SW Approach'),
    mpatches.Patch(color='#a0ff80', label='Northbound'),
    mpatches.Patch(color='#ff80d5', label='Southbound'),
    mpatches.Patch(color='#ffdd00', label='Anchored'),
]
ax.legend(handles=legend_items, loc='lower left', fontsize=7.5,
          framealpha=0.45, facecolor='black', edgecolor='#445566',
          labelcolor='white', handlelength=1.2, borderpad=0.6, labelspacing=0.35)

# Scale bar (~10 km)
sb_x0, sb_y0 = transformer.transform(103.88, 1.165)
sb_x1, _     = transformer.transform(103.98, 1.165)
sb_y = sb_y0
tick = (ymax - ymin) * 0.008
ax.plot([sb_x0, sb_x1], [sb_y, sb_y], lw=2.5, color='white',
        solid_capstyle='butt', zorder=9)
ax.plot([sb_x0, sb_x0], [sb_y - tick, sb_y + tick], lw=2, color='white', zorder=9)
ax.plot([sb_x1, sb_x1], [sb_y - tick, sb_y + tick], lw=2, color='white', zorder=9)
ax.text((sb_x0 + sb_x1) / 2, sb_y - tick * 1.8, '~10 km',
        ha='center', va='top', fontsize=7.5, color='white', family='monospace',
        path_effects=[pe.withStroke(linewidth=1.5, foreground='black')], zorder=9)

# ═══════════════════════════════════════════════════════════════════════════════
# 8. UPDATE FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

def update(frame):
    global trail_lines
    for ln in trail_lines:
        ln.remove()
    trail_lines = []

    pts = {vt: [] for vt in STYLE}

    for i, v in enumerate(vessels):
        xs, ys = merc_coords[i]
        if np.isnan(xs[frame]):
            continue
        vtype = v['type']
        pts[vtype].append((xs[frame], ys[frame]))

        if vtype != 'anchored':
            start = max(0, frame - TRAIL)
            tx = xs[start:frame + 1]
            ty = ys[start:frame + 1]
            valid = ~np.isnan(tx)
            if valid.sum() > 1:
                alpha = max(0.08, 0.55 * (valid.sum() / TRAIL))
                ln, = ax.plot(tx[valid], ty[valid],
                              color=STYLE[vtype]['color'],
                              lw=0.9, alpha=alpha, zorder=2)
                trail_lines.append(ln)

    for vtype, sc in scatters.items():
        p = pts[vtype]
        sc.set_offsets(np.array(p) if p else np.empty((0, 2)))

    # Clock
    sim_h = SIM_START + (frame / N_FRAMES) * SIM_HOURS
    hh, mm = int(sim_h) % 24, int((sim_h % 1) * 60)
    clock_txt.set_text(f'SGT {hh:02d}:{mm:02d}')

    # Counts
    counts = {t: len(pts[t]) for t in STYLE}
    total  = sum(counts.values())
    count_txt.set_text(
        f'E:{counts["eastbound"]}  W:{counts["westbound"]}  '
        f'CR:{counts["crossing"] + counts["northbound"] + counts["southbound"]}  '
        f'A:{counts["anchored"]}  │  {total} vessels'
    )
    return [*scatters.values(), *trail_lines, clock_txt, count_txt]

# ═══════════════════════════════════════════════════════════════════════════════
# 9. RENDER TO MP4
# ═══════════════════════════════════════════════════════════════════════════════

OUTPUT_PATH = '/tmp/singapore_strait_vessels.mp4'

writer = FFMpegWriter(
    fps=FPS, codec='libx264', bitrate=4500,
    extra_args=['-pix_fmt', 'yuv420p', '-preset', 'fast',
                '-crf', '18', '-movflags', '+faststart']
)

print(f'\n🎬 Rendering {N_FRAMES} frames → {DURATION_S}s MP4 …')
t0 = time.time()

with writer.saving(fig, OUTPUT_PATH, dpi=DPI):
    for frame in tqdm(range(N_FRAMES), desc='Rendering', unit='frame'):
        update(frame)
        writer.grab_frame()

elapsed = time.time() - t0
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f'\n✅  Done in {elapsed / 60:.1f} min  →  {size_mb:.1f} MB')
files.download(OUTPUT_PATH)

⬇️  Downloading basemap (Esri World Imagery via contextily)…
✅  Basemap saved  1280×768 px
Total frames: 4320  (180s × 24fps)  |  102 vessels
Vessels spawned: 102
⚙️  Pre-computing Mercator coordinates…
✅  Done

🎬 Rendering 4320 frames → 180s MP4 …


Rendering:   0%|          | 0/4320 [00:00<?, ?frame/s]


✅  Done in 14.3 min  →  10.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGAPORE STRAIT — AIS DISRUPTION EVENT + DARK VESSEL OSINT SIMULATION
# Free basemap (Esri via contextily) | 2-min MP4 | Dramatic AIS shutdown story
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'contextily', 'pyproj', 'numpy', 'matplotlib', 'tqdm'], check=True)
subprocess.run(['apt-get', 'install', '-q', '-y', 'ffmpeg'], check=True)

import io, os, math, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.animation import FFMpegWriter
from matplotlib.patches import FancyArrowPatch
from pyproj import Transformer
import contextily as ctx
from tqdm.auto import tqdm
from google.colab import files

# ═══════════════════════════════════════════════════════════════════════════════
# 1. MAP EXTENT
# ═══════════════════════════════════════════════════════════════════════════════

LON_MIN, LON_MAX = 103.55, 104.35
LAT_MIN, LAT_MAX = 1.10,  1.45

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
xmin, ymin = transformer.transform(LON_MIN, LAT_MIN)
xmax, ymax = transformer.transform(LON_MAX, LAT_MAX)

def to_merc(lons, lats):
    return transformer.transform(lons, lats)

# ═══════════════════════════════════════════════════════════════════════════════
# 2. BASEMAP
# ═══════════════════════════════════════════════════════════════════════════════

MAP_CACHE = '/tmp/sg_basemap_osm.npz'

if os.path.exists(MAP_CACHE):
    data      = np.load(MAP_CACHE)
    basemap   = data['img']
    bm_extent = list(data['extent'])
    print(f'✅  Loaded cached basemap  {basemap.shape[1]}×{basemap.shape[0]} px')
else:
    print('⬇️  Downloading basemap…')
    fig_tmp, ax_tmp = plt.subplots(figsize=(12, 7))
    ax_tmp.set_xlim(xmin, xmax); ax_tmp.set_ylim(ymin, ymax)
    ctx.add_basemap(ax_tmp, source=ctx.providers.Esri.WorldImagery,
                    zoom=11, crs='EPSG:3857')
    imgs = [c for c in ax_tmp.get_children()
            if isinstance(c, matplotlib.image.AxesImage)]
    bm   = imgs[0]
    arr  = bm.get_array()
    basemap   = (arr * 255).astype(np.uint8) if arr.max() <= 1.0 else arr.astype(np.uint8)
    bm_extent = list(bm.get_extent())
    plt.close(fig_tmp)
    np.savez_compressed(MAP_CACHE, img=basemap, extent=np.array(bm_extent))
    print(f'✅  Basemap saved  {basemap.shape[1]}×{basemap.shape[0]} px')

# ═══════════════════════════════════════════════════════════════════════════════
# 3. STORY TIMELINE  (frame numbers)
#
#   Act 1  [0  → F*0.30]  Normal busy traffic — ~80 vessels visible
#   Act 2  [F*0.30 → F*0.38]  AIS SHUTDOWN EVENT — vessels vanish rapidly
#   Act 3  [F*0.38 → F*0.55]  Ghost strait — only handful of vessels remain
#   Act 4  [F*0.55 → F*0.65]  Dark vessel (LPG tanker) sneaks through — NO AIS dot
#   Act 5  [F*0.65 → F*0.78]  Dark vessel re-activates AIS on far side
#   Act 6  [F*0.78 → 1.0  ]  Slow traffic resumption
# ═══════════════════════════════════════════════════════════════════════════════

DURATION_S = 120
FPS        = 24
N_FRAMES   = DURATION_S * FPS   # 2 880
TRAIL      = FPS * 5

T_SHUTDOWN_START = int(N_FRAMES * 0.30)
T_SHUTDOWN_END   = int(N_FRAMES * 0.38)
T_GHOST_END      = int(N_FRAMES * 0.55)
T_DARK_END       = int(N_FRAMES * 0.65)
T_REAPPEAR       = int(N_FRAMES * 0.66)
T_RESUME         = int(N_FRAMES * 0.78)

SIM_HOURS = 8.0
SIM_START = 4.0   # 04:00 SGT

rng = np.random.default_rng(7)

# ═══════════════════════════════════════════════════════════════════════════════
# 4. TRACK HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def make_track(lon0, lat0, lon1, lat1,
               spread_lon=0.0, spread_lat=0.010,
               speed_var=0.12, stagger=0, wobble=0.0004):
    dlon = rng.uniform(-spread_lon, spread_lon) if spread_lon else 0
    dlat = rng.uniform(-spread_lat, spread_lat)
    speed = rng.normal(1.0, speed_var, N_FRAMES).clip(0.4, 1.8)
    cum   = np.cumsum(speed)
    t     = (cum - cum[0]) / (cum[-1] - cum[0])
    lon   = (lon0 + dlon) + t * (lon1 - lon0)
    lat   = (lat0 + dlat) + t * (lat1 - lat0)
    phase = rng.uniform(0, 2 * np.pi)
    side  = np.sin(2 * np.pi * rng.uniform(0.2, 0.6) * t + phase) * wobble
    dx, dy = lon1 - lon0, lat1 - lat0
    norm = math.hypot(dx, dy) or 1
    lon += side * (-dy / norm); lat += side * (dx / norm)
    if stagger > 0:
        lon = np.roll(lon, stagger); lon[:stagger] = np.nan
        lat = np.roll(lat, stagger); lat[:stagger] = np.nan
    return lon, lat

def make_anchored(alon, alat):
    dl = np.cumsum(rng.normal(0, 0.00004, N_FRAMES))
    da = np.cumsum(rng.normal(0, 0.00003, N_FRAMES))
    return alon + dl, alat + da

# ═══════════════════════════════════════════════════════════════════════════════
# 5. VESSEL FLEET
#    Each vessel carries an extra field:  ais_off_start / ais_off_end
#    When frame is inside [ais_off_start, ais_off_end] the dot is hidden.
# ═══════════════════════════════════════════════════════════════════════════════

vessels = []

# ── Normal fleet (active during Act 1, shut off during Act 2/3) ───────────────

# Eastbound TSS (22)
for _ in range(22):
    stag = int(rng.integers(0, N_FRAMES // 3))
    l, a = make_track(103.58, 1.168, 104.28, 1.178,
                      spread_lat=0.011, stagger=stag)
    # AIS goes dark between shutdown start and ghost end, with slight random spread
    off_s = T_SHUTDOWN_START + int(rng.integers(0, int(N_FRAMES * 0.06)))
    off_e = T_GHOST_END      + int(rng.integers(0, int(N_FRAMES * 0.10)))
    vessels.append({'type':'eastbound','lon':l,'lat':a,
                    'ais_off_start':off_s,'ais_off_end':off_e})

# Westbound TSS (22)
for _ in range(22):
    stag = int(rng.integers(0, N_FRAMES // 3))
    l, a = make_track(104.28, 1.210, 103.58, 1.200,
                      spread_lat=0.011, stagger=stag)
    off_s = T_SHUTDOWN_START + int(rng.integers(0, int(N_FRAMES * 0.06)))
    off_e = T_GHOST_END      + int(rng.integers(0, int(N_FRAMES * 0.10)))
    vessels.append({'type':'westbound','lon':l,'lat':a,
                    'ais_off_start':off_s,'ais_off_end':off_e})

# Crossing / SW Approach (8)
for _ in range(8):
    stag = int(rng.integers(0, N_FRAMES // 3))
    l, a = make_track(103.58, 1.230, 103.88, 1.190,
                      spread_lon=0.01, spread_lat=0.008, stagger=stag)
    off_s = T_SHUTDOWN_START + int(rng.integers(0, int(N_FRAMES * 0.07)))
    off_e = T_GHOST_END      + int(rng.integers(0, int(N_FRAMES * 0.12)))
    vessels.append({'type':'crossing','lon':l,'lat':a,
                    'ais_off_start':off_s,'ais_off_end':off_e})

# Northbound (4)
for _ in range(4):
    stag = int(rng.integers(0, N_FRAMES // 3))
    l, a = make_track(103.95 + rng.uniform(-0.04,0.04), 1.155,
                      103.93 + rng.uniform(-0.04,0.04), 1.330,
                      spread_lon=0.015, spread_lat=0.0, stagger=stag)
    off_s = T_SHUTDOWN_START + int(rng.integers(0, int(N_FRAMES * 0.07)))
    off_e = T_GHOST_END      + int(rng.integers(0, int(N_FRAMES * 0.12)))
    vessels.append({'type':'northbound','lon':l,'lat':a,
                    'ais_off_start':off_s,'ais_off_end':off_e})

# Southbound (4)
for _ in range(4):
    stag = int(rng.integers(0, N_FRAMES // 3))
    l, a = make_track(103.73 + rng.uniform(-0.04,0.04), 1.330,
                      103.71 + rng.uniform(-0.04,0.04), 1.155,
                      spread_lon=0.015, spread_lat=0.0, stagger=stag)
    off_s = T_SHUTDOWN_START + int(rng.integers(0, int(N_FRAMES * 0.07)))
    off_e = T_GHOST_END      + int(rng.integers(0, int(N_FRAMES * 0.12)))
    vessels.append({'type':'southbound','lon':l,'lat':a,
                    'ais_off_start':off_s,'ais_off_end':off_e})

# Anchored (12) — stay visible throughout (they don't transit)
for _ in range(12):
    al = rng.uniform(104.00, 104.20); aa = rng.uniform(1.220, 1.270)
    l, a = make_anchored(al, aa)
    vessels.append({'type':'anchored','lon':l,'lat':a,
                    'ais_off_start':999999,'ais_off_end':999999})

# ── THE DARK VESSEL — LPG tanker sneaks through ────────────────────────────────
# Travels full west→east but AIS is OFF from T_GHOST_END to T_REAPPEAR
# We give it a special type so we can draw the "ghost" track and re-appear flash

dark_lon, dark_lat = make_track(103.58, 1.185, 104.28, 1.190,
                                spread_lat=0.004, wobble=0.0002)
vessels.append({
    'type'          : 'dark_vessel',
    'lon'           : dark_lon,
    'lat'           : dark_lat,
    'ais_off_start' : T_GHOST_END,       # goes dark entering chokepoint
    'ais_off_end'   : T_REAPPEAR,        # re-appears on far side
})

DARK_IDX = len(vessels) - 1
print(f'Fleet size: {len(vessels)} vessels  (including 1 dark vessel)')

# ═══════════════════════════════════════════════════════════════════════════════
# 6. PRE-COMPUTE MERCATOR
# ═══════════════════════════════════════════════════════════════════════════════

print('⚙️  Pre-computing Mercator coordinates…')
merc = []
for v in vessels:
    lons, lats = v['lon'], v['lat']
    valid = ~np.isnan(lons)
    xs = np.full(N_FRAMES, np.nan)
    ys = np.full(N_FRAMES, np.nan)
    if valid.any():
        xs[valid], ys[valid] = to_merc(lons[valid], lats[valid])
    merc.append((xs, ys))
print('✅  Done')

# ═══════════════════════════════════════════════════════════════════════════════
# 7. STYLE
# ═══════════════════════════════════════════════════════════════════════════════

STYLE = {
    'eastbound'  : {'color':'#00cfff','size':26,'marker':'>','zorder':5},
    'westbound'  : {'color':'#ff6b35','size':26,'marker':'<','zorder':5},
    'crossing'   : {'color':'#ffe066','size':20,'marker':'^','zorder':4},
    'northbound' : {'color':'#a0ff80','size':20,'marker':'^','zorder':4},
    'southbound' : {'color':'#ff80d5','size':20,'marker':'v','zorder':4},
    'anchored'   : {'color':'#ffdd00','size':18,'marker':'s','zorder':3},
    'dark_vessel': {'color':'#ff2222','size':55,'marker':'*','zorder':9},
}

# ═══════════════════════════════════════════════════════════════════════════════
# 8. FIGURE
# ═══════════════════════════════════════════════════════════════════════════════

DPI = 110
fig, ax = plt.subplots(figsize=(1280/DPI, 720/DPI), dpi=DPI, facecolor='black')
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
ax.axis('off')

ax.imshow(basemap, extent=bm_extent, origin='upper', aspect='auto', zorder=0)

# ── Scatter artists (one per type) ────────────────────────────────────────────
scatters = {}
for vtype, st in STYLE.items():
    scatters[vtype] = ax.scatter(
        [], [], s=st['size'], c=st['color'],
        marker=st['marker'], zorder=st['zorder'],
        edgecolors='white', linewidths=0.3, alpha=0.92
    )

# ── Ghost trail for dark vessel (dashed, red) ─────────────────────────────────
ghost_line, = ax.plot([], [], color='#ff4444', lw=1.2,
                      linestyle='--', alpha=0.0, zorder=8)

# ── "AIS RE-ACQUIRED" flash ring ──────────────────────────────────────────────
flash_ring = plt.Circle((0, 0), radius=0, color='#ff2222',
                         fill=False, lw=2.5, alpha=0.0, zorder=10)
ax.add_patch(flash_ring)

trail_lines = []

# ── Red overlay for AIS shutdown event ───────────────────────────────────────
# (full-screen tinted rectangle that fades in/out)
red_overlay = plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                             color='red', alpha=0.0, zorder=6)
ax.add_patch(red_overlay)

# ── HUD elements ──────────────────────────────────────────────────────────────
title_txt = ax.text(0.5, 0.977,
    'SINGAPORE STRAIT — AIS VESSEL TRACKING',
    transform=ax.transAxes, ha='center', va='top',
    fontsize=13, fontweight='bold', color='white', family='monospace',
    path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=11)

clock_txt = ax.text(0.013, 0.977, '',
    transform=ax.transAxes, ha='left', va='top',
    fontsize=9, color='#a0ffcc', family='monospace',
    path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=11)

count_txt = ax.text(0.987, 0.977, '',
    transform=ax.transAxes, ha='right', va='top',
    fontsize=8.5, color='white', family='monospace',
    path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=11)

# Big event alert banner (hidden by default)
alert_txt = ax.text(0.5, 0.50, '',
    transform=ax.transAxes, ha='center', va='center',
    fontsize=16, fontweight='bold', color='#ff3333', family='monospace',
    alpha=0.0,
    path_effects=[pe.withStroke(linewidth=3, foreground='black')], zorder=12)

# Dark vessel label
dark_label = ax.text(0.0, 0.0, '',
    ha='left', va='bottom',
    fontsize=7.5, color='#ff9999', family='monospace',
    path_effects=[pe.withStroke(linewidth=1.5, foreground='black')], zorder=12)

# Legend
legend_items = [
    mpatches.Patch(color='#00cfff', label='Eastbound'),
    mpatches.Patch(color='#ff6b35', label='Westbound'),
    mpatches.Patch(color='#ffe066', label='SW Approach'),
    mpatches.Patch(color='#a0ff80', label='Northbound'),
    mpatches.Patch(color='#ff80d5', label='Southbound'),
    mpatches.Patch(color='#ffdd00', label='Anchored'),
    mpatches.Patch(color='#ff2222', label='⬛ DARK VESSEL'),
]
ax.legend(handles=legend_items, loc='lower left', fontsize=7,
          framealpha=0.50, facecolor='black', edgecolor='#ff2222',
          labelcolor='white', handlelength=1.2, borderpad=0.6, labelspacing=0.35)

# Scale bar
sb_x0, sb_y0 = transformer.transform(103.88, 1.135)
sb_x1, _     = transformer.transform(103.98, 1.135)
tick = (ymax - ymin) * 0.008
ax.plot([sb_x0,sb_x1],[sb_y0,sb_y0], lw=2.5, color='white',
        solid_capstyle='butt', zorder=9)
ax.plot([sb_x0,sb_x0],[sb_y0-tick,sb_y0+tick], lw=2, color='white', zorder=9)
ax.plot([sb_x1,sb_x1],[sb_y0-tick,sb_y0+tick], lw=2, color='white', zorder=9)
ax.text((sb_x0+sb_x1)/2, sb_y0-tick*1.8, '~10 km',
        ha='center', va='top', fontsize=7.5, color='white', family='monospace',
        path_effects=[pe.withStroke(linewidth=1.5, foreground='black')], zorder=9)

# ── Act label (bottom centre) ─────────────────────────────────────────────────
act_txt = ax.text(0.5, 0.030, '',
    transform=ax.transAxes, ha='center', va='bottom',
    fontsize=8, color='#aaaacc', family='monospace', alpha=0.85,
    path_effects=[pe.withStroke(linewidth=1.5, foreground='black')], zorder=11)

# ═══════════════════════════════════════════════════════════════════════════════
# 9. UPDATE
# ═══════════════════════════════════════════════════════════════════════════════

def smooth_step(x):
    """Smooth 0→1 transition."""
    x = np.clip(x, 0, 1)
    return x * x * (3 - 2 * x)

def update(frame):
    global trail_lines

    for ln in trail_lines:
        ln.remove()
    trail_lines = []

    pts = {vt: [] for vt in STYLE}

    # ── AIS shutdown progress ──────────────────────────────────────────────────
    if frame < T_SHUTDOWN_START:
        shutdown_frac = 0.0
    elif frame < T_SHUTDOWN_END:
        shutdown_frac = smooth_step((frame - T_SHUTDOWN_START) /
                                    (T_SHUTDOWN_END - T_SHUTDOWN_START))
    elif frame < T_RESUME:
        shutdown_frac = 1.0
    else:
        shutdown_frac = 1.0 - smooth_step((frame - T_RESUME) /
                                           (N_FRAMES - T_RESUME))

    # Red overlay intensity (subtle)
    red_alpha = shutdown_frac * 0.13
    red_overlay.set_alpha(red_alpha)

    # ── Iterate vessels ────────────────────────────────────────────────────────
    for i, v in enumerate(vessels):
        xs, ys = merc[i]
        if np.isnan(xs[frame]):
            continue

        vtype  = v['type']
        off_s  = v['ais_off_start']
        off_e  = v['ais_off_end']
        is_ais_dark = (off_s <= frame <= off_e)

        if vtype == 'dark_vessel':
            # Handled separately below
            continue

        if is_ais_dark:
            continue   # vessel is invisible on AIS

        pts[vtype].append((xs[frame], ys[frame]))

        # Wake trail
        if vtype != 'anchored':
            start = max(0, frame - TRAIL)
            tx = xs[start:frame+1]; ty = ys[start:frame+1]
            valid = ~np.isnan(tx)
            if valid.sum() > 1:
                alpha = max(0.07, 0.5 * (valid.sum() / TRAIL))
                ln, = ax.plot(tx[valid], ty[valid],
                              color=STYLE[vtype]['color'],
                              lw=0.8, alpha=alpha, zorder=2)
                trail_lines.append(ln)

    # ── Dark vessel logic ──────────────────────────────────────────────────────
    dv       = vessels[DARK_IDX]
    dxs, dys = merc[DARK_IDX]
    off_s    = dv['ais_off_start']
    off_e    = dv['ais_off_end']
    is_dark  = (off_s <= frame <= off_e)
    dx_now   = dxs[frame] if not np.isnan(dxs[frame]) else None

    if dx_now is not None:
        if not is_dark:
            pts['dark_vessel'].append((dxs[frame], dys[frame]))

            # Normal trail when AIS is on
            start = max(0, frame - TRAIL)
            tx = dxs[start:frame+1]; ty = dys[start:frame+1]
            # Only show trail when AIS was on
            mask = np.array([not (off_s <= f <= off_e)
                             for f in range(start, frame+1)])
            valid = ~np.isnan(tx) & mask
            if valid.sum() > 1:
                ln, = ax.plot(tx[valid], ty[valid],
                              color='#ff2222', lw=1.2, alpha=0.6, zorder=7)
                trail_lines.append(ln)

        else:
            # While dark: draw ghost dotted trail (faint red dashes)
            start = max(off_s, frame - TRAIL * 2)
            tx = dxs[start:frame+1]; ty = dys[start:frame+1]
            valid = ~np.isnan(tx)
            if valid.sum() > 1:
                ghost_line.set_data(tx[valid], ty[valid])
                ghost_line.set_alpha(0.35)
            else:
                ghost_line.set_alpha(0.0)

    if not is_dark:
        ghost_line.set_alpha(0.0)

    # ── Re-appear flash ring ───────────────────────────────────────────────────
    flash_alpha = 0.0
    flash_radius = 0
    if T_REAPPEAR <= frame <= T_REAPPEAR + FPS * 2:
        t_flash = (frame - T_REAPPEAR) / (FPS * 2)
        flash_alpha  = max(0, 1.0 - t_flash * 1.1)
        flash_radius = t_flash * (xmax - xmin) * 0.04
        if dx_now is not None:
            flash_ring.center = (dxs[frame], dys[frame])
    flash_ring.set_radius(flash_radius)
    flash_ring.set_alpha(flash_alpha)

    # ── Dark vessel label ──────────────────────────────────────────────────────
    if dx_now is not None and frame >= off_s - FPS * 3:
        ax_x = (dxs[frame] - xmin) / (xmax - xmin)
        ax_y = (dys[frame] - ymin) / (ymax - ymin)
        if is_dark:
            dark_label.set_text('⬛ AIS DARK\nMV STAR ORIENT\n(LPG TANKER)')
            dark_label.set_color('#ff9999')
        elif frame >= T_REAPPEAR:
            dark_label.set_text('✅ AIS RE-ACQUIRED\nMV STAR ORIENT')
            dark_label.set_color('#00ff99')
        else:
            dark_label.set_text('MV STAR ORIENT\n(tracking)')
            dark_label.set_color('#ffcccc')
        dark_label.set_position((dxs[frame] + (xmax-xmin)*0.01,
                                  dys[frame] + (ymax-ymin)*0.015))
        dark_label.set_alpha(1.0)
    else:
        dark_label.set_alpha(0.0)

    # ── Update scatters ────────────────────────────────────────────────────────
    for vtype, sc in scatters.items():
        p = pts[vtype]
        sc.set_offsets(np.array(p) if p else np.empty((0, 2)))

    # ── Clock ──────────────────────────────────────────────────────────────────
    sim_h = SIM_START + (frame / N_FRAMES) * SIM_HOURS
    hh, mm = int(sim_h) % 24, int((sim_h % 1) * 60)
    clock_txt.set_text(f'SGT {hh:02d}:{mm:02d}')

    # ── Vessel count ───────────────────────────────────────────────────────────
    total = sum(len(v) for v in pts.values())
    count_txt.set_text(
        f'AIS CONTACTS: {total}  │  '
        f'E:{len(pts["eastbound"])}  W:{len(pts["westbound"])}  '
        f'A:{len(pts["anchored"])}'
    )

    # ── Alert banner ───────────────────────────────────────────────────────────
    if T_SHUTDOWN_START <= frame <= T_SHUTDOWN_END:
        prog = int(shutdown_frac * 100)
        alert_txt.set_text(f'⚠  MASS AIS DROPOUT DETECTED  —  {prog}% SIGNAL LOSS')
        alert_txt.set_alpha(min(1.0, shutdown_frac * 3))
        alert_txt.set_color('#ff3333')
    elif T_SHUTDOWN_END < frame < T_GHOST_END:
        alert_txt.set_text('⚠  STRAIT GONE DARK  —  MINIMAL AIS COVERAGE')
        alert_txt.set_alpha(0.85)
        alert_txt.set_color('#ff6600')
    elif T_GHOST_END <= frame < T_DARK_END:
        alert_txt.set_text('🔴  DARK VESSEL TRANSITING CHOKEPOINT  —  NO AIS')
        alert_txt.set_alpha(0.95)
        alert_txt.set_color('#ff2222')
    elif T_REAPPEAR <= frame <= T_REAPPEAR + FPS * 3:
        alert_txt.set_text('📡  AIS RE-ACQUIRED  —  MV STAR ORIENT  —  POSITION FIXED')
        alert_txt.set_alpha(1.0)
        alert_txt.set_color('#00ff99')
    elif frame >= T_RESUME:
        resume_frac = (frame - T_RESUME) / (N_FRAMES - T_RESUME)
        alert_txt.set_text('✅  AIS SIGNALS RECOVERING…')
        alert_txt.set_alpha(max(0, 0.7 - resume_frac))
        alert_txt.set_color('#aaffaa')
    else:
        alert_txt.set_alpha(0.0)

    # ── Act label ──────────────────────────────────────────────────────────────
    if frame < T_SHUTDOWN_START:
        act_txt.set_text('ACT I — NORMAL OPERATIONS  |  HIGH VESSEL DENSITY')
    elif frame < T_SHUTDOWN_END:
        act_txt.set_text('ACT II — AIS SHUTDOWN EVENT IN PROGRESS')
    elif frame < T_GHOST_END:
        act_txt.set_text('ACT III — GHOST STRAIT  |  SURVEILLANCE BLACKOUT')
    elif frame < T_REAPPEAR:
        act_txt.set_text('ACT IV — DARK VESSEL TRANSIT  |  MV STAR ORIENT RUNNING SILENT')
    elif frame < T_RESUME:
        act_txt.set_text('ACT V — RE-ACQUISITION  |  DARK VESSEL EMERGES ON FAR SIDE')
    else:
        act_txt.set_text('ACT VI — SIGNAL RECOVERY  |  TRAFFIC RESUMING')

    return [*scatters.values(), *trail_lines,
            ghost_line, flash_ring, red_overlay,
            clock_txt, count_txt, alert_txt, act_txt, dark_label]

# ═══════════════════════════════════════════════════════════════════════════════
# 10. RENDER
# ═══════════════════════════════════════════════════════════════════════════════

OUTPUT_PATH = '/tmp/sg_ais_disruption.mp4'

writer = FFMpegWriter(
    fps=FPS, codec='libx264', bitrate=5000,
    extra_args=['-pix_fmt','yuv420p','-preset','fast',
                '-crf','17','-movflags','+faststart']
)

print(f'\n🎬 Rendering {N_FRAMES} frames → {DURATION_S}s MP4 …')
t0 = time.time()

with writer.saving(fig, OUTPUT_PATH, dpi=DPI):
    for frame in tqdm(range(N_FRAMES), desc='Rendering', unit='frame'):
        update(frame)
        writer.grab_frame()

elapsed = time.time() - t0
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f'\n✅  Done in {elapsed/60:.1f} min  →  {size_mb:.1f} MB')
files.download(OUTPUT_PATH)

⬇️  Downloading basemap…
✅  Basemap saved  1280×768 px
Fleet size: 73 vessels  (including 1 dark vessel)
⚙️  Pre-computing Mercator coordinates…
✅  Done

🎬 Rendering 2880 frames → 120s MP4 …


Rendering:   0%|          | 0/2880 [00:00<?, ?frame/s]

/tmp/ipykernel_6479/3860995704.py:555: UserWarning: Glyph 11035 (\N{BLACK LARGE SQUARE}) missing from font(s) DejaVu Sans.
  writer.grab_frame()
/tmp/ipykernel_6479/3860995704.py:555: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans Mono.
  writer.grab_frame()
/tmp/ipykernel_6479/3860995704.py:555: UserWarning: Glyph 11035 (\N{BLACK LARGE SQUARE}) missing from font(s) DejaVu Sans Mono.
  writer.grab_frame()
/tmp/ipykernel_6479/3860995704.py:555: UserWarning: Glyph 128225 (\N{SATELLITE ANTENNA}) missing from font(s) DejaVu Sans Mono.
  writer.grab_frame()
/tmp/ipykernel_6479/3860995704.py:555: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans Mono.
  writer.grab_frame()



✅  Done in 10.0 min  →  9.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 11035 (\N{BLACK LARGE SQUARE}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
